In [73]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate
import collections
import pickle,os,csv
from pathlib import Path  
import itertools
import seaborn as sns
import glob
import os
from scipy import stats

In [74]:
# import models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# import model evaluation metrics
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import imblearn
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score


In [75]:
# import datasets

synonoms = 'drugbank_vocabulary.csv'
drug_syn = pd.read_csv(synonoms)
drug_sider = pd.read_csv('drug_names.tsv', sep='\t')
drug_SE = pd.read_csv('meddra_all_se.tsv', sep='\t')
DB_summary = pd.read_csv('drugbank.tsv', sep='\t')

In [76]:
# Generate drug names to DBID dictionary 
# ensures that both common names and synonyms of a drug can be used to look up its DrugBank ID

drug_syn['synlist'] = ""
for i in range(0, len(drug_syn['Synonyms'])):
    drug_syn.loc[i, 'synlist'] = ", ".join(str(drug_syn.loc[i, 'Synonyms']).lower().split(" | "))
    drug_syn.loc[i, 'Common name'] = drug_syn.loc[i, 'Common name'].lower()

from collections import defaultdict
chemical_to_DBID=defaultdict(set)


for (drug, DBID) in zip(drug_syn['Common name'], drug_syn['DrugBank ID']):
  
  chemical_to_DBID[drug].add(DBID)

for(drug, DBID) in zip(drug_syn['synlist'], drug_syn['DrugBank ID']):
  for i in range(0, len(drug)):
    chemical_to_DBID[drug[i]].add(DBID)

In [77]:
##SIDER drug name dataset processing##
# Standardizes drug names and links SIDER drugs to DrugBank for integration

#Rearrange rows and rename columns

drug_sider.loc[-1] = drug_sider.columns.values
drug_sider.sort_index(inplace=True)
drug_sider.reset_index(drop=True, inplace=True)
drug_sider.columns=['drugID', 'drugname']

#Map SIDER drugnames to DBID

for i in range(len(drug_sider['drugname'])):
    drug_sider.loc[i, 'drugname']=drug_sider.loc[i, 'drugname'].lower()

drug_sider['DBID'] = ''
drug_sider['DBID'] = drug_sider['drugname'].map(chemical_to_DBID)

for i in range(len(drug_sider['DBID'])):
    drug_sider.loc[i, 'DBID'] = str(drug_sider.loc[i, 'DBID']).replace('{',"").replace('}',"").replace("'","")
    
#Determine number of DBID match

match_bool=drug_sider['DBID'] !='set()'
match_index = match_bool[match_bool].index

print('The number of matched DBIDS is', sum(drug_sider['DBID'] !='set()'))
print('The number of unmatched DBIDS is', sum(drug_sider['DBID'] =='set()'))

print(drug_sider.head())

The number of matched DBIDS is 1095
The number of unmatched DBIDS is 335
         drugID                  drugname     DBID
0  CID100000085                 carnitine    set()
1  CID100000119        gamma-aminobutyric    set()
2  CID100000137          5-aminolevulinic    set()
3  CID100000143                leucovorin  DB00650
4  CID100000146  5-methyltetrahydrofolate    set()


In [78]:
# Generate SIDER DrugID to DBID - this will map drug_SE drugID to its respective DBID
drugID_to_DBID={}

for i in range(len(drug_sider['drugID'])):
    drugID_to_DBID[drug_sider.loc[i, 'drugID']] = drug_sider.loc[i, 'DBID']

In [79]:
# run to get drug_SE data frame
drug_SE = pd.read_excel('./intermediate_data/drug_side_effects.xlsx')

In [80]:
# Generate the list of top 30 side effects in SIDER 4.1
from collections import Counter

side_effect_count = pd.DataFrame(Counter(drug_SE['side_effect']).most_common(30), columns=['Side Effect', 'Drug Count',])

In [81]:
#Generate Dictionary of top 30 side effects with its associated DrugBank ID

from collections import defaultdict

sid_to_dbid = defaultdict(list)
for j in side_effect_count['Side Effect']:
    for i in range(len(drug_SE['side_effect'])):
        if drug_SE['side_effect'][i] == j:
            sid_to_dbid['phen_ind_'+str(j)].append(drug_SE['DBID'][i])

for key, item in sid_to_dbid.items():
    sid_to_dbid[key] = list(set([x for x in item if str(x) !='set()']))

# Functions

In [82]:
#Matrix Generation

def matrix(dic):
    all_rows=[]
    for (drug, dt_set) in dic.items():
        row_data = {'DrugName': drug}
        for dt in dt_set:
            row_data[dt] = 1
        all_rows.append(row_data)
    dt_df = pd.DataFrame(all_rows)
    dt_df = dt_df.fillna(0)
    dt_df = dt_df[dt_df['DrugName'].str.contains('DB')]
    return dt_df

In [83]:
# feature matrix for approved drugs only
def matrix_approved(dic):
    approved_index = []
    all_rows=[]
    for (drug, dt_set) in dic.items():
        row_data = {'DrugName': drug}
        for dt in dt_set:
            row_data[dt] = 1
        all_rows.append(row_data)
    dt_df = pd.DataFrame(all_rows)
    dt_df = dt_df.fillna(0)
    dt_df = dt_df[dt_df['DrugName'].str.contains('DB')]
    for i, k in enumerate(dt_df['DrugName']):
        for j in approved_drugs:
            if k == j:
                approved_index.append(i)
    dt_df_approved = dt_df.iloc[approved_index]
    return dt_df_approved

In [84]:
#Model Inputs
# Transforms the dataset into a machine-learning-ready format, 
# where X is the feature set, and sid_eff_pred contains labels for predicting side effects
def model_input(matrix):
    for key, value in sid_to_dbid.items():
        matrix[key]=''
    for key, value in sid_to_dbid.items():
        for i in value:
            matrix.loc[matrix['DrugName'] != i, key] = 0
    for key, value in sid_to_dbid.items():
        for i in value:
            matrix.loc[matrix['DrugName'] == i, key] = 1
    key_all = []
    for key, value in sid_to_dbid.items():
        key_all.append(key)
        
    X = matrix.drop(key_all, axis=1)
    X = X.drop(['DrugName'], axis=1).values
    
    sid_eff_pred = {}
    for j in side_effect_count['Side Effect']:
        sid_eff_pred[j] = matrix['phen_ind_'+str(j)].astype('int')
    
    return X, sid_eff_pred

In [85]:
#Logistic Regression Bootstrapped 100 Times

# Runs the model multiple times 
# splits 80% training, 20% testing
# trains a model for each side effect
# predicts and stores accuracy scores in boot_df

def log_reg_boot100(X, Y):
    bootstrapNum = 100
    boot_df = pd.DataFrame()
    for k , j in Y.items():
        for i in range (bootstrapNum):
            log_reg = LogisticRegression()
            rus = RandomUnderSampler() # balance classes so model learns to recognize features that actually lead to the side effect, instead of just defaulting to "0" (majority class)
            X_sample, Y_sample = rus.fit_resample(X, j)
            X_train, X_test, Y_train, Y_test = train_test_split(X_sample, Y_sample, train_size = 0.8, test_size=0.2, random_state=1)
            log_reg.fit(X_train, Y_train)
            Y_pred = log_reg.predict(X_test)
            boot_df = pd.concat((boot_df, pd.DataFrame({k: accuracy_score(Y_test, Y_pred)}, index=[i])))
    boot_df = boot_df.apply(lambda x: pd.Series(x.dropna().values))
    return boot_df

In [86]:
# Isolate approved drugs and their targets 
a2n = pickle.load(open('Pfx050120_dint.pkl', 'rb'))

# modified below (caused a key error before)
for i in range(len(DB_summary['groups'])):
    DB_summary['groups'][i] = DB_summary['groups'][i].split('|')

approved_drugs = set()
approved_dbids = {DB_summary['drugbank_id'][i] for i, group in enumerate(DB_summary['groups']) for drug in group if drug == 'approved'}
for dbid, targets in a2n.items():
    if dbid in approved_dbids:
        approved_drugs.add(dbid)

/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_22575/429280400.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  DB_summary['groups'][i] = DB_summary['groups'][i].split('|')


In [87]:
# creates separate datasets for all drugs vs. approved drugs to compare side effect predictions
targets_approved = matrix_approved(a2n)
targets = matrix(a2n)

X, Y = model_input(targets)
X_approved, Y_approved = model_input(targets_approved)

targets_approved.head()

,DrugName,MAPK10,astB,CYP2B6,TTR,SLC6A2,ABCB1,ABCG2,CYP2D6,SLC6A3,...,phen_ind_insomnia,phen_ind_anaphylactic shock,phen_ind_paraesthesia,phen_ind_somnolence,phen_ind_nervous system disorder,phen_ind_thrombocytopenia,phen_ind_tachycardia,phen_ind_arthralgia,phen_ind_infection,phen_ind_musculoskeletal discomfort
4,DB00285,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,...,1,1,1,1,0,1,1,1,1,1
7,DB00648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,1,1,1,1,0,0,1,0
16,DB00043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
17,DB00417,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
22,DB09097,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [88]:
# shows how many drugs were mapped to a side effect

side_effect_count['DBID Match Count'] = ''
# side_effect_count['Matrix Match Count'] = ''
for i, j in enumerate(side_effect_count['Side Effect']):
    side_effect_count.loc[i, 'DBID Match Count'] = len(sid_to_dbid['phen_ind_'+str(j)])

for i, j in enumerate(side_effect_count['Side Effect']):
    side_effect_count.loc[i, 'Matrix Match Count'] = targets_approved['phen_ind_'+str(j)].value_counts()[1]

# ATC Levels Only Comparison

In [89]:
# ATC Codes ONLY
# load in Level 2

ATC_only = pd.read_excel("./intermediate_data/atc_only.xlsx")

In [90]:
# Model Evaluation with ATC Level 2 Codes Only

X, Y = model_input(ATC_only)
LR_ATC = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC[k].mean())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

This is the LR mean accuracy on x100 bootstrap for dizziness 0.6773898305084747
This is the LR mean accuracy on x100 bootstrap for nausea 0.6823353293413171
This is the LR mean accuracy on x100 bootstrap for headache 0.666012658227848
This is the LR mean accuracy on x100 bootstrap for rash 0.659503311258278
This is the LR mean accuracy on x100 bootstrap for vomiting 0.6980592105263157
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7041947565543072
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.6954347826086957
This is the LR mean accuracy on x100 bootstrap for pruritus 0.6971641791044776
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.67804
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.6914042553191488
This is the LR mean accuracy on x100 bootstrap for urticaria 0.6808771929824563
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.7125
This is the LR mean accuracy on x100 bootstrap f

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

In [95]:
# ATC Codes ONLY
# load in Levels 2, 3, 4, 5

# ATC_only = pd.read_excel("./intermediate_data/atc_only.xlsx")
ATC_only_3 = pd.read_excel("./intermediate_data/atc_only_3.xlsx")
ATC_only_4 = pd.read_excel("./intermediate_data/atc_only_4.xlsx")
ATC_only_5 = pd.read_excel("./intermediate_data/atc_only_5.xlsx")

In [91]:
# # ATC Codes ONLY
# # repeat for Level 3
    
# # Create DBID to ATC dictionary
# DBatc = defaultdict(set)

# for dbid, targets in a2n.items(): # changed to a2n because omitting PathFX, hopefully this is correct someone please double check
#     for i in range(len(DB_summary['atc_codes'])):
#         DB_atc = set()
#         if dbid == DB_summary['drugbank_id'][i]:
#             for j in range(len(DB_summary['atc_codes'][i])):
#                 DB_atc.add(DB_summary['atc_codes'][i][j][0:4])
#             DBatc[DB_summary['drugbank_id'][i]] = DB_atc

# # Generate Matrix
# ATC_only_3 = matrix_approved(DBatc)
# ATC_only_3 = ATC_only_3[ATC_only_3['DrugName'].str.contains('DB')]
# ATC_only_3 = ATC_only_3.drop(columns = ['N/A'])
# ATC_only_3

In [92]:
ATC_only_3.to_excel("atc_only_3.xlsx") # saving intermediate data

In [93]:
# Model Evaluation with ATC Level 3 Codes Only

X, Y = model_input(ATC_only_3)
LR_ATC_3 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC_3[k].mean())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

This is the LR mean accuracy on x100 bootstrap for dizziness 0.6656610169491525
This is the LR mean accuracy on x100 bootstrap for nausea 0.6680538922155687
This is the LR mean accuracy on x100 bootstrap for headache 0.6776582278481013
This is the LR mean accuracy on x100 bootstrap for rash 0.672913907284768
This is the LR mean accuracy on x100 bootstrap for vomiting 0.6765460526315791
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7077902621722848
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.6873550724637683
This is the LR mean accuracy on x100 bootstrap for pruritus 0.7077611940298509
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.6819600000000001
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.6938297872340425
This is the LR mean accuracy on x100 bootstrap for urticaria 0.692324561403509
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.6831250000000001
This is the LR mean accur

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

In [ ]:
# ATC Codes ONLY
# repeat for Level 4
    
# Create DBID to ATC dictionary
DBatc4 = defaultdict(set)

for dbid, targets in a2n.items(): # changed to a2n because omitting PathFX, hopefully this is correct someone please double check
    for i in range(len(DB_summary['atc_codes'])):
        DB_atc = set()
        if dbid == DB_summary['drugbank_id'][i]:
            for j in range(len(DB_summary['atc_codes'][i])):
                DB_atc.add(DB_summary['atc_codes'][i][j][0:5])
            DBatc4[DB_summary['drugbank_id'][i]] = DB_atc

# Generate Matrix
ATC_only_4 = matrix_approved(DBatc4)
ATC_only_4 = ATC_only_4[ATC_only_4['DrugName'].str.contains('DB')]
ATC_only_4 = ATC_only_4.drop(columns = ['N/A'])
ATC_only_4

ATC_only_4.to_excel("atc_only_4.xlsx") # saving intermediate data


KeyError: 'DrugName'

In [96]:
# Model Evaluation with ATC Level 4 Codes Only

X, Y = model_input(ATC_only_4)
LR_ATC_4 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC_4[k].mean())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

This is the LR mean accuracy on x100 bootstrap for dizziness 0.6760338983050848
This is the LR mean accuracy on x100 bootstrap for nausea 0.6814071856287424
This is the LR mean accuracy on x100 bootstrap for headache 0.6801265822784809
This is the LR mean accuracy on x100 bootstrap for rash 0.6746688741721854
This is the LR mean accuracy on x100 bootstrap for vomiting 0.6599671052631579
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7113857677902624
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.6813405797101452
This is the LR mean accuracy on x100 bootstrap for pruritus 0.7092164179104478
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.6630399999999999
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.6828510638297872
This is the LR mean accuracy on x100 bootstrap for urticaria 0.6981578947368422
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.6959374999999999
This is the LR mean acc

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# # ATC Codes ONLY
# # repeat for Level 5
    
# # Create DBID to ATC dictionary
# DBatc = defaultdict(set)

# for dbid, targets in a2n.items(): # changed to a2n because omitting PathFX, hopefully this is correct someone please double check
#     for i in range(len(DB_summary['atc_codes'])):
#         DB_atc = set()
#         if dbid == DB_summary['drugbank_id'][i]:
#             for j in range(len(DB_summary['atc_codes'][i])):
#                 DB_atc.add(DB_summary['atc_codes'][i][j][0:7])
#             DBatc[DB_summary['drugbank_id'][i]] = DB_atc

# # Generate Matrix
# ATC_only_5 = matrix_approved(DBatc)
# ATC_only_5 = ATC_only_5[ATC_only_5['DrugName'].str.contains('DB')]
# ATC_only_5 = ATC_only_5.drop(columns = ['N/A'])
# ATC_only_5

# ATC_only_5.to_excel("atc_only_5.xlsx") # saving intermediate data

In [ ]:
# Model Evaluation with ATC Level 5 Codes Only

X, Y = model_input(ATC_only_5)
LR_ATC_5 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC_5[k].mean())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

In [ ]:
# from statsmodels.stats.anova import AnovaRM
# import pandas as pd

# lr_ANOVA = pd.DataFrame()

# for j, i in enumerate(side_effect_count['Side Effect']):
#     ANOVA = pd.DataFrame()
#     ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC[i], 'condition': 'LR: ATC Level 2'})))
#     ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC_3[i], 'condition': 'LR: ATC Level 3'})))
#     ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC_4[i], 'condition': 'LR: ATC Level 4'})))
#     ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC_5[i], 'condition': 'LR: ATC Level 5'})))
    
#     ANOVA.reset_index(inplace=True)
#     results = AnovaRM(data=ANOVA, depvar='model accuracy', subject='index', within=['condition']).fit()

#     lr_ANOVA = pd.concat((lr_ANOVA, pd.DataFrame({
#         'Side Effect': i, 
#         'LR: ATC Level 2': LR_ATC[i].mean(),
#         'LR: ATC Level 3': LR_ATC_3[i].mean(),
#         'LR: ATC Level 4': LR_ATC_4[i].mean(),
#         'LR: ATC Level 5': LR_ATC_5[i].mean(),
#         'F-Value': results.anova_table['F Value'][0], 
#         'P-value': results.anova_table['Pr > F'][0]
#     }, index=[j])))

# lr_ANOVA.to_excel("LR_ANOVA_ATC_Level234.xlsx")
# lr_ANOVA


/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_22575/3016437219.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'F-Value': results.anova_table['F Value'][0],
/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_22575/3016437219.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'P-value': results.anova_table['Pr > F'][0]
/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_22575/3016437219.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a 

,Side Effect,LR: ATC Level 2,LR: ATC Level 3,LR: ATC Level 4,LR: ATC Level 5,F-Value,P-value
0,dizziness,0.675898,0.667797,0.679627,0.539119,1168.720528,4.728380e-164
1,nausea,0.683144,0.668892,0.683263,0.556317,1232.911303,3.090303e-167
2,headache,0.665696,0.680222,0.684778,0.570506,728.472149,1.504487e-136
3,rash,0.664735,0.670497,0.674503,0.565497,798.056049,9.382497e-142
4,vomiting,0.696447,0.673816,0.663224,0.526283,1532.945956,2.460005e-180
5,asthenia,0.707678,0.711124,0.708202,0.546816,1499.301749,5.421531e-179
6,diarrhoea,0.690797,0.684710,0.681087,0.544203,1168.512581,4.844948e-164
7,pruritus,0.700075,0.710336,0.703321,0.586679,747.621707,5.041405e-138
8,hypersensitivity,0.683280,0.679480,0.661400,0.582000,511.390292,6.171078e-117
9,abdominal pain,0.698894,0.698979,0.689745,0.582213,488.473645,1.805859e-114


# ATC and Drug Targets Comparison

In [ ]:
# adding drug targets as features
tar_ATC = pd.read_excel("./intermediate_data/tar_atc.xlsx")

print(tar_ATC.head())

   Unnamed: 0 DrugName  MAPK10  astB  CYP2B6  TTR  SLC6A4  ABCB1  ABCG2  \
0           4  DB00285       0     0       0    0       1      1      1   
1           7  DB00648       0     0       0    0       0      0      0   
2          16  DB00043       0     0       0    0       0      0      0   
3          17  DB00417       0     0       0    0       0      0      0   
4          22  DB09097       0     0       0    0       0      0      0   

   CYP2D6  ...  malT  oxyR  RIDA  Cyp2b2  Cyp2b9  Cyp2a2  KEAP1  RELA  V06  \
0       1  ...     0     0     0       0       0       0      0     0    0   
1       0  ...     0     0     0       0       0       0      0     0    0   
2       0  ...     0     0     0       0       0       0      0     0    0   
3       0  ...     0     0     0       0       0       0      0     0    0   
4       0  ...     0     0     0       0       0       0      0     0    0   

   trpS2  
0      0  
1      0  
2      0  
3      0  
4      0  

[5 rows x 390

In [ ]:
# Model Evaluation with Drugbank Targets and ATC Codes

# X, Y = model_input(tar_ATC)
# LR_Tar_ATC = log_reg_boot100(X, Y)
# for k , j in Y.items():
#     print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_Tar_ATC[k].mean())

LR_Tar_ATC = pd.read_excel("./intermediate_data/LR_Tar_ATC.xlsx")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

This is the LR mean accuracy on x100 bootstrap for dizziness 0.695762711864407
This is the LR mean accuracy on x100 bootstrap for nausea 0.6827844311377244
This is the LR mean accuracy on x100 bootstrap for headache 0.6865189873417721
This is the LR mean accuracy on x100 bootstrap for rash 0.6694701986754964
This is the LR mean accuracy on x100 bootstrap for vomiting 0.709342105263158
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7002621722846443
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.7081521739130437
This is the LR mean accuracy on x100 bootstrap for pruritus 0.685783582089552
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.7021600000000001
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.7164255319148936
This is the LR mean accuracy on x100 bootstrap for urticaria 0.6932017543859648
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.7113392857142857
This is the LR mean accura

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
tar_atc_4 = matrix_approved(DBatc)
X, Y = model_input(tar_ATC_4)
LR_Tar_ATC_4 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_Tar_ATC_4[k].mean())
LR_Tar_ATC_4.to_excel('LR_Tar_ATC.xlsx')

In [ ]:
from statsmodels.stats.anova import AnovaRM

# Initialize an empty DataFrame to store results
lr_ANOVA = pd.DataFrame()

# Loop over side effects to perform the ANOVA comparison
for j, i in enumerate(side_effect_count['Side Effect']):
    # Initialize an empty DataFrame for each ANOVA comparison
    ANOVA = pd.DataFrame()
    
    # Add model accuracy for different conditions
    # First condition: 'LR: ATC Only'
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC[i], 'condition':'LR: ATC Only'})))
    
    # Second condition: 'LR: Drug Targets with ATC Level 2'
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_tar_approved[i], 'condition':'LR: Drug Targets with ATC Level 2'})))
    
    # Third condition: 'LR: Drug Targets with ATC Level 4'
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_Tar_ATC[i], 'condition':'LR: Drug Targets with ATC Level 4'})))
    
    # Reset index and perform ANOVA
    ANOVA.reset_index(inplace=True)
    results = AnovaRM(data=ANOVA, depvar='model accuracy', subject='index', within=['condition']).fit()
    
    # Append the ANOVA results to the final DataFrame
    lr_ANOVA = pd.concat((lr_ANOVA, pd.DataFrame({
        'Side Effect': i, 
        'LR: ATC Only': LR_ATC[i].mean(),
        'LR: Drug Targets with ATC Level 2': LR_tar_approved[i].mean(),
        'LR: Drug Targets with ATC Level 4': LR_Tar_ATC[i].mean(),
        'F-Value': results.anova_table['F Value'][0], 
        'P-value': results.anova_table['Pr > F'][0]
    }, index=[j])))

# Save the ANOVA results to an Excel file
lr_ANOVA.to_excel("LR_ANOVA.xlsx")  

# Display the results
lr_ANOVA
